# YOLO11n fine-tune on Roboflow COCO — Kaggle pipeline

**Why Kaggle, not Colab.** Kaggle gives 12h continuous GPU sessions, persistent `/kaggle/working` between save+run, and free P100/T4 quotas. We also save a *Resume* artefact every epoch so a killed session can be restarted from the last checkpoint instead of from scratch.

**Settings before running** (right sidebar in the Kaggle editor):
1. **Accelerator** → GPU P100 (preferred) or T4 x2.
2. **Internet** → On (needed for Roboflow download and pip).
3. **Add-ons → Secrets** → add `ROBOFLOW_API_KEY` (do NOT paste it in a cell).

If a session times out: open the notebook, change `RESUME = True`, hit *Save Version → Save & Run All*. Training continues from `last.pt` in the Kaggle dataset attached as input.

In [ ]:
!pip -q install --upgrade ultralytics roboflow onnx onnxslim onnxruntime coremltools tensorflow

In [ ]:
import os, shutil, sys, json, time, subprocess, pathlib
from pathlib import Path

WORKSPACE = 'microsoft'
PROJECT   = 'coco'
VERSION   = 50
MODEL     = 'yolo11n.pt'
EPOCHS    = 30
IMGSZ     = 640
BATCH     = 16
RUN_NAME  = 'coco_yolo11n'

RESUME    = False  # flip to True to resume from a previous Kaggle run
PRIOR_RUN_INPUT = '/kaggle/input/yolo11n-coco-run'  # dataset name if RESUME

WORK = Path('/kaggle/working')
RUNS = WORK / 'runs/detect'
RUNS.mkdir(parents=True, exist_ok=True)
print('GPU info:')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'no nvidia-smi'

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ['ROBOFLOW_API_KEY'] = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')

In [ ]:
from roboflow import Roboflow
os.chdir(WORK)
rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download('yolov8')
DATA_YAML = str(Path(dataset.location) / 'data.yaml')
print('data.yaml:', DATA_YAML)
!head -n 20 {DATA_YAML}

In [ ]:
# Resume from a previous Kaggle run (attached as an input dataset). Optional.
if RESUME:
    prior = Path(PRIOR_RUN_INPUT)
    target = RUNS / RUN_NAME
    target.mkdir(parents=True, exist_ok=True)
    shutil.copytree(prior, target, dirs_exist_ok=True)
    print('Restored prior run into', target)
    !ls {target}/weights

In [ ]:
from ultralytics import YOLO
from ultralytics import settings as ul_settings
ul_settings.update({'datasets_dir': str(WORK)})

start_weights = MODEL
if RESUME:
    last = RUNS / RUN_NAME / 'weights' / 'last.pt'
    if last.exists():
        start_weights = str(last)
        print('Resuming from', start_weights)

model = YOLO(start_weights)
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    project=str(RUNS),
    name=RUN_NAME,
    exist_ok=True,
    resume=RESUME,
    patience=20,
    save=True,
    save_period=1,        # checkpoint every epoch so a kill is recoverable
    plots=True,
    seed=42,
    deterministic=False,
)
BEST = Path(results.save_dir) / 'weights' / 'best.pt'
print('best:', BEST)

In [ ]:
val = model.val(data=DATA_YAML, imgsz=IMGSZ, plots=True)
metrics = {
    'mAP50_95': float(val.box.map),
    'mAP50'   : float(val.box.map50),
    'precision': float(val.box.mp),
    'recall'   : float(val.box.mr),
}
(Path(results.save_dir) / 'final_metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

In [ ]:
# Export every mobile format. INT8 uses dataset for calibration.
best_model = YOLO(str(BEST))
exports = {}
exports['onnx']   = best_model.export(format='onnx',   imgsz=IMGSZ, opset=12, simplify=True)
exports['tflite'] = best_model.export(format='tflite', imgsz=IMGSZ, int8=True, data=DATA_YAML, nms=True)
exports['coreml'] = best_model.export(format='coreml', imgsz=IMGSZ, int8=True, data=DATA_YAML, nms=True)
for k, v in exports.items():
    size_mb = Path(v).stat().st_size / 1e6 if Path(v).exists() else 0
    print(f'{k:7s} -> {v} ({size_mb:.2f} MB)')

In [ ]:
# Collect the small, important artefacts so they survive Save Version.
ART = WORK / 'artifacts'
ART.mkdir(exist_ok=True)
for p in [BEST, *exports.values()]:
    p = Path(p)
    if p.exists():
        if p.is_dir():
            shutil.copytree(p, ART / p.name, dirs_exist_ok=True)
        else:
            shutil.copy2(p, ART / p.name)
shutil.copy2(Path(results.save_dir) / 'final_metrics.json', ART / 'final_metrics.json')
print('Artifacts ready in', ART)
!ls -lh {ART}

### Saving a resumable checkpoint as a Kaggle dataset
After this notebook finishes (or if you commit/save partway), go to *Output* → *New Dataset* and publish `runs/detect/coco_yolo11n/` as a dataset (e.g. `yolo11n-coco-run`). On the next run, attach that dataset as an input, set `RESUME = True`, and re-run all. Training picks up at the last saved epoch.